# Liu's TEC-rate + MW dual-frequency cycle-slip method

**Course goal:** detect and *repair* real integer L1/L2 carrier jumps with
two observables: **Melbourne–Wübbena** and **time derivative of ionospheric
TEC**. This is especially useful for understanding small, equal slips and
complementary detector sensitivities.

Z. Liu's article appeared **online in 2010**, and in *Journal of Geodesy*
**85 (2011), 171–183**. This notebook uses a simplified transparent
implementation inspired by it, not an exact numerical GIPSY/TurboEdit clone.

Reference: [Liu (2010/2011), *A new automated cycle slip detection and repair method for a single dual-frequency GPS receiver*](https://doi.org/10.1007/s00190-010-0426-y).

**Sampling warning:** the published 0.15 TECU/s criterion was tested at
approximately **1 Hz**. Your shared DYNG data are every **30 s**; we must
not assert equal sensitivity there. We first do a reproducible 1 Hz exercise.


In [ ]:
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from dsoclasses.gnss.cycle_slip_common import (
    teaching_gps_series, gps_series_from_rinex, combinations,
)

data_path = "../../../data"
rinex_path = join(data_path, "DYNG00GRC_R_20240010000_01D_30S_MO.rnx")

# A reproducible 1 Hz example with known ground truth (not a measured orbit).
clean = teaching_gps_series(n=420, seed=2024)
truth = {120: (3, -2), 240: (1, 1), 330: (77, 60)}
corrupted = clean.with_changes(
    slips=truth,
    code_outliers={300: (6., 0.)},    # one bad code observation
    phase_outliers={370: (5., 0.)},   # one bad carrier epoch, NOT a cycle slip
)
t = corrupted.t_s
print(f'GPS L1 wavelength = {clean.wavelength1_m:.6f} m')
print(f'GPS L2 wavelength = {clean.wavelength2_m:.6f} m')
print(f'Wide-lane wavelength = {clean.wavelength_wide_m:.6f} m')
print(f'GF phase = {clean.gf_metres_per_tecu:.6f} m / TECU (after a fixed ambiguity)')
print(f'Simulated phase slips: {truth}; isolated outliers at t = 300, 370 s')


## 1. Measurements and two complementary combinations

Two simultaneous *code* measurements $P_1,P_2$ [metres] and *carrier*
measurements $L_1,L_2$ [cycles], tracked with **the same signal types**
throughout one arc, are enough to detect many cycle slips. We do **not**
need orbit interpolation or a tropospheric model for this first exercise.

The Melbourne–Wübbena wide-lane combination [cycles]:

$$MW = L_1-L_2 - \frac{f_1P_1+f_2P_2}{(f_1+f_2)\lambda_W}, \quad
\lambda_W=\frac{c}{f_1-f_2}. $$

The geometry-free phase combination [metres]:

$$GF=\lambda_1L_1-\lambda_2L_2,\quad \lambda_i=c/f_i.$$

For a genuine jump $(\Delta N_1,\Delta N_2)$, after removing smooth
ionospheric evolution:

$$\Delta MW \approx \Delta N_1-\Delta N_2,\qquad
\Delta GF \approx \lambda_1\Delta N_1-\lambda_2\Delta N_2.$$

**Why both?** Equal integer slips barely affect MW, while particular
large slip pairs nearly cancel in GF. A bad code sample can affect MW
without affecting GF. A single bad phase sample can look like a slip for
one epoch but does *not* persist.


In [ ]:
mw_clean, gf_clean = combinations(clean)
mw_bad, gf_bad = combinations(corrupted)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, mw_bad, '.', ms=3, label='Observed MW (corrupted)')
ax[0].plot(t, mw_clean, color='tab:green', lw=1.3, label='Uncorrupted truth')
ax[0].set(ylabel='MW [cycles]', title='Raw combinations: slips, noise and isolated errors')
ax[1].plot(t, gf_bad, '.', ms=3, label='Observed GF (corrupted)')
ax[1].plot(t, gf_clean, color='tab:green', lw=1.3, label='Uncorrupted truth')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='GF [m]')
for a in ax:
    for index in truth:
        a.axvline(index, c='tab:red', alpha=.32, lw=1)
    for index in (300, 370):
        a.axvline(index, c='tab:orange', alpha=.5, lw=1, ls='--')
    a.grid(alpha=.25)
    a.legend(loc='best')
plt.tight_layout()
plt.show()
print('Red lines: persistent slips. Dashed orange: isolated code/phase outliers.')


## 2. From geometry-free carrier phase to TEC rate

The first-order ionospheric group delay for signal $f$ is
$40.3\cdot10^{16}\,\mathrm{TEC}/f^2$ [m] when TEC is in TECU. The GF carrier
combination is proportional to TEC plus a **constant ambiguity**:

$$GF(t) \approx K\,\mathrm{TEC}(t)+B, \qquad
K=\frac{40.3\cdot10^{16}}{f_1^2}\left[\left(\frac{f_1}{f_2}\right)^2-1\right].$$

Take differences: the unknown constant $B$ cancels:

$$\mathrm{TECR}_{obs}(t_i)\approx
\frac{GF(t_i)-GF(t_{i-1})}{K(t_i-t_{i-1})}.$$

Liu's idea is to estimate the **next expected TEC rate** from preceding
clean rates (about 30 epochs), accounting for a smooth trend/acceleration.
Then compare $\mathrm{TECR}_{obs}-\mathrm{TECR}_{pred}$ with the threshold.
An abrupt jump in GF now appears as an anomalous TEC rate, while MW
provides a second integer constraint.


In [ ]:
from dsoclasses.gnss.liu2010 import liu2010

result = liu2010(corrupted, rate_history=30, tecr_threshold_tecu_s=.15)
tecr_error = result.tecr_observed - result.tecr_predicted
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, result.tecr_observed, '.', alpha=.35, ms=3, label='Observed TEC rate')
ax[0].plot(t, result.tecr_predicted, color='tab:orange', lw=1.4,
           label='Predicted TEC rate from earlier 30 samples')
ax[0].set(ylabel='TECU/s', title='TEC-rate prediction and slip innovation')
ax[1].plot(t, tecr_error, '.', ms=3, label='TEC-rate innovation')
ax[1].axhline(+.15, color='grey', ls='--', label='published ±0.15 TECU/s')
ax[1].axhline(-.15, color='grey', ls='--')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='Observed - predicted [TECU/s]')
for a in ax:
    for index in truth: a.axvline(index, color='tab:red', alpha=.25)
    a.legend(loc='best')
    a.grid(alpha=.25)
plt.tight_layout()
plt.show()
print('Samples before rate prediction starts:', np.isnan(result.tecr_predicted).sum())
print('Nominal GF threshold at 1 Hz [m]:', .15*clean.gf_metres_per_tecu)


## 3. MW supplies the complementary integer equation

When MW and the corrected GF/TEC-rate prediction show a persistent step,
we search for $(\Delta N_1,\Delta N_2)$ satisfying:

$$\Delta N_1-\Delta N_2\approx\Delta MW, \qquad
\lambda_1\Delta N_1-\lambda_2\Delta N_2\approx\Delta GF_{detrended}.$$

The solver rejects ambiguous pairs; it **does not blindly round**.
The one-epoch look-ahead to distinguish a one-time outlier from a permanent
slip is an *offline addition* to this simplified classroom implementation.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, result.mw_innovation, '.', ms=3, label='MW innovation')
ax[0].axhline(+.45, color='grey', ls='--', label='±0.45 cycles nominal floor')
ax[0].axhline(-.45, color='grey', ls='--')
ax[0].set(ylabel='Cycles', title='MW vs. TEC-rate: complementary detectors')
ax[1].plot(t, tecr_error, '.', ms=3, label='TEC-rate innovation')
ax[1].axhline(+.15, color='grey', ls='--', label='±0.15 TECU/s published reference')
ax[1].axhline(-.15, color='grey', ls='--')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='TECU/s')
for a in ax:
    for index in truth: a.axvline(index, color='tab:red', alpha=.25)
    a.grid(alpha=.25)
    a.legend()
plt.tight_layout()
plt.show()


## 4. Print each anomaly and its two-frequency repair

Notice: a $(1,1)$ slip is **invisible to MW**, while the $(77,60)$ slip is
almost invisible to the GF/TEC-rate test. The two constraints recover both.
An isolated code outlier affects MW; a one-time phase outlier does not
persist. A disputed repair would be reported as `unresolved` and the phase
ambiguity arc broken.


In [ ]:
print(' epoch  classification       ΔMW [cy]   ΔGF [m]    ΔN1    ΔN2')
print('-' * 72)
for event in result.events:
    s1 = '-' if event.slip_l1 is None else str(event.slip_l1)
    s2 = '-' if event.slip_l2 is None else str(event.slip_l2)
    print(f'{event.index:5d}  {event.kind:18s} '
          f'{event.mw_step_cycles:+9.3f}  {event.gf_step_m:+9.4f} '
          f'{s1:>6s} {s2:>6s}')

assert [(e.index, e.slip_l1, e.slip_l2) for e in result.slips] == [
    (120, 3, -2), (240, 1, 1), (330, 77, 60)]
assert result.status[300] == 'code_outlier'
assert result.status[370] == 'phase_outlier'
print('Check passed: correct integers recovered; isolated errors were not repaired as slips.')


In [ ]:
# Work out ONE integer repair by hand, using just the plotted MW/GF steps.
event = next(e for e in result.slips if e.index == 120)
wide_lane_integer = round(event.mw_step_cycles)
n2_float = ((event.gf_step_m - clean.wavelength1_m*wide_lane_integer) /
            (clean.wavelength1_m - clean.wavelength2_m))
n1_float = n2_float + wide_lane_integer
print('At epoch 120:')
print(f'  ΔMW = {event.mw_step_cycles:+.3f} cycles -> ΔN1 - ΔN2 = {wide_lane_integer}')
print(f'  ΔGF = {event.gf_step_m:+.4f} metres')
print(f'  estimated (ΔN1, ΔN2) before rounding = ({n1_float:+.3f}, {n2_float:+.3f})')
print(f'  accepted checked integer pair = ({event.slip_l1:+d}, {event.slip_l2:+d})')
print('  Caution: the module searches and validates competing integer pairs;')
print('  merely rounding this demonstration is NOT a reliable repair rule.')


## 5. Verify repaired carrier phases against known truth

The correction changes only carrier-phase *integers*. Code spikes cannot
be repaired that way. Both the suspect code epoch and the isolated phase
epoch remain excluded, ready for later phase positioning.


In [ ]:
# Apply only the accepted integer repairs. Bad epochs are NaN and stay excluded.
corrected_mw = (result.mw_cycles -
                (corrupted.l1_cycles - result.repaired_l1_cycles) +
                (corrupted.l2_cycles - result.repaired_l2_cycles))
corrected_gf = (result.gf_m -
                clean.wavelength1_m * (corrupted.l1_cycles - result.repaired_l1_cycles) +
                clean.wavelength2_m * (corrupted.l2_cycles - result.repaired_l2_cycles))

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, mw_bad - mw_clean, '.', ms=3, label='Before: MW error')
ax[0].plot(t, corrected_mw - mw_clean, '.', ms=2, label='After: MW error')
ax[0].set(ylabel='MW - truth [cycles]', title='Effect of accepted integer repairs')
ax[1].plot(t, gf_bad - gf_clean, '.', ms=3, label='Before: GF error')
ax[1].plot(t, corrected_gf - gf_clean, '.', ms=2, label='After: GF error')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='GF - truth [m]')
for a in ax:
    a.grid(alpha=.25)
    a.legend()
plt.tight_layout()
plt.show()

valid = np.isfinite(result.repaired_l1_cycles)
print('Maximum carrier L1 difference to known clean truth, after repair [cycles]:',
      np.max(np.abs(result.repaired_l1_cycles[valid] - clean.l1_cycles[valid])))
print('Excluded epochs:', [(i, s) for i, s in enumerate(result.status)
                          if s in ('phase_outlier', 'code_outlier', 'unresolved')])


## 6. Real DYNG RINEX: observe, do not overclaim

The class RINEX is sampled at 30-second intervals; the 1 Hz TEC-rate
threshold is **not transferable unchanged**. This cell still teaches you
to find a consistent RINEX signal pair, compute MW and GF, and see the
actual cadence. It deliberately **does not** claim validated Liu repairs
from these sparse observations. Data lookup follows the class path convention.


In [ ]:
try:
    real = gps_series_from_rinex(rinex_path, satellite='G21', max_samples=350)
except (FileNotFoundError, LookupError) as exc:
    print('Real RINEX demonstration not available:', exc)
else:
    real_mw, real_gf = combinations(real)
    print('Fixed GPS descriptors:', real.signals)
    print('Median sample spacing [s]:', np.median(np.diff(real.t_s)))
    fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    ax[0].plot(real.t_s/60, real_mw, '.', ms=2)
    ax[0].set(ylabel='MW [cycles]', title='Actual DYNG G21 observations')
    ax[1].plot(real.t_s/60, real_gf, '.', ms=2)
    ax[1].set(xlabel='Minutes after first selected observation', ylabel='GF [m]')
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout()
    plt.show()
    try:
        liu2010(real)
    except ValueError as exc:
        print('Expected safety check:', exc)
